# Homework. Comparison of classification algorithms

Dataset: `airline_passenger_satisfaction.csv`. Task: predict passenger satisfaction and compare Random Forest, KNeighborsClassifier and Logistic Regression.

## Data preparation

In [1]:
import time

import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, fbeta_score, confusion_matrix

In [2]:
df = pd.read_csv('airline_passenger_satisfaction.csv')
df.drop(columns=['id', 'Unnamed: 0'], inplace=True)
df.dropna(inplace=True)

label_encoder = LabelEncoder()
columns = ['Gender', 'Type of Travel', 'Customer Type', 'Class']
for col in columns:
    df[col] = label_encoder.fit_transform(df[col])

df.head()

,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,1,0,13,1,2,460,3,4,3,1,...,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,1,25,0,0,235,3,2,3,3,...,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,0,0,26,0,0,1142,2,2,2,2,...,5,4,3,4,4,4,5,0,0.0,satisfied
3,0,0,25,0,0,562,2,5,5,5,...,2,2,5,3,1,4,2,11,9.0,neutral or dissatisfied
4,1,0,61,0,0,214,3,3,3,3,...,3,3,4,4,3,3,3,0,0.0,satisfied


In [3]:
X = df.drop(columns='satisfaction')
y = df['satisfaction']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")

Train size: 82875
Test size:  20719


## Evaluation function

One function for all three models, so they are measured the same way. Each model is trained on `X_train` only and checked on `X_test`, which it has never seen.

In [4]:
def evaluate_model(name, model, X_train, X_test):
    start = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start

    start = time.time()
    predictions = model.predict(X_test)
    predict_time = time.time() - start

    cm = confusion_matrix(y_test, predictions)

    print(name)
    print("=" * 55)
    print(f"Accuracy:                {accuracy_score(y_test, predictions):.4f}")
    print(f"Precision (satisfied):   {precision_score(y_test, predictions, pos_label='satisfied'):.4f}")
    print(f"Recall (satisfied):      {recall_score(y_test, predictions, pos_label='satisfied'):.4f}")
    print(f"F1-Score:                {f1_score(y_test, predictions, pos_label='satisfied'):.4f}")
    print(f"F-Beta 0.5 (precision):  {fbeta_score(y_test, predictions, beta=0.5, average='weighted'):.4f}")
    print(f"F-Beta 2.0 (recall):     {fbeta_score(y_test, predictions, beta=2.0, average='weighted'):.4f}")
    print(f"Training time:           {train_time:.1f} sec")
    print(f"Prediction time:         {predict_time:.1f} sec")
    print("")
    print(f"True Negative  (correctly 'dissatisfied'): {cm[0, 0]}")
    print(f"False Positive (wrongly 'satisfied'):      {cm[0, 1]}")
    print(f"False Negative (wrongly 'dissatisfied'):   {cm[1, 0]}")
    print(f"True Positive  (correctly 'satisfied'):    {cm[1, 1]}")

    return {
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, pos_label='satisfied'),
        'Recall': recall_score(y_test, predictions, pos_label='satisfied'),
        'F1': f1_score(y_test, predictions, pos_label='satisfied'),
        'Train time': train_time,
        'Predict time': predict_time
    }

## Random Forest

In [5]:
results = []
results.append(evaluate_model('Random Forest', RandomForestClassifier(random_state=42), X_train, X_test))

Random Forest
Accuracy:                0.9628
Precision (satisfied):   0.9712
Recall (satisfied):      0.9422
F1-Score:                0.9565
F-Beta 0.5 (precision):  0.9629
F-Beta 2.0 (recall):     0.9628
Training time:           5.1 sec
Prediction time:         0.2 sec

True Negative  (correctly 'dissatisfied'): 11489
False Positive (wrongly 'satisfied'):      251
False Negative (wrongly 'dissatisfied'):   519
True Positive  (correctly 'satisfied'):    8460


## KNeighborsClassifier

In [6]:
results.append(evaluate_model('K Neighbors', KNeighborsClassifier(n_neighbors=5), X_train, X_test))

K Neighbors
Accuracy:                0.7414
Precision (satisfied):   0.7196
Recall (satisfied):      0.6610
F1-Score:                0.6890
F-Beta 0.5 (precision):  0.7398


F-Beta 2.0 (recall):     0.7406
Training time:           0.1 sec
Prediction time:         0.6 sec

True Negative  (correctly 'dissatisfied'): 9427
False Positive (wrongly 'satisfied'):      2313
False Negative (wrongly 'dissatisfied'):   3044
True Positive  (correctly 'satisfied'):    5935


## Logistic Regression

In [7]:
results.append(evaluate_model('Logistic Regression', LogisticRegression(max_iter=10000), X_train, X_test))

Logistic Regression
Accuracy:                0.8720
Precision (satisfied):   0.8675
Recall (satisfied):      0.8317
F1-Score:                0.8492
F-Beta 0.5 (precision):  0.8717
F-Beta 2.0 (recall):     0.8718
Training time:           16.8 sec
Prediction time:         0.0 sec

True Negative  (correctly 'dissatisfied'): 10599
False Positive (wrongly 'satisfied'):      1141
False Negative (wrongly 'dissatisfied'):   1511
True Positive  (correctly 'satisfied'):    7468


## Comparison

In [8]:
pd.DataFrame(results)

,Model,Accuracy,Precision,Recall,F1,Train time,Predict time
0,Random Forest,0.962836,0.971186,0.942198,0.956473,5.083063,0.187304
1,K Neighbors,0.741445,0.719568,0.660987,0.689035,0.062959,0.604885
2,Logistic Regression,0.872002,0.867464,0.831718,0.849215,16.750859,0.001517


Random Forest is clearly the best. K Neighbors is much worse than the other two, which is strange for this algorithm, so the next part checks why.

## Why K Neighbors is the weakest

K Neighbors finds passengers that are close to each other. If one column has much bigger numbers than the others, only that column decides who is close.

In [9]:
scales = X.describe().loc[['min', 'max']].T
scales['range'] = scales['max'] - scales['min']
scales.sort_values('range', ascending=False).head(5)

,min,max,range
Flight Distance,31.0,4983.0,4952.0
Departure Delay in Minutes,0.0,1592.0,1592.0
Arrival Delay in Minutes,0.0,1584.0,1584.0
Age,7.0,85.0,78.0
Seat comfort,0.0,5.0,5.0


Flight Distance goes up to 4983 and delays up to 1592 minutes, while service ratings are only from 0 to 5. So the distance between two passengers was decided by kilometres, not by their ratings.

`StandardScaler` puts all columns on the same scale.

In [10]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scaled_results = []
scaled_results.append(evaluate_model('Random Forest (scaled)', RandomForestClassifier(random_state=42), X_train_scaled, X_test_scaled))

Random Forest (scaled)
Accuracy:                0.9629
Precision (satisfied):   0.9714
Recall (satisfied):      0.9421
F1-Score:                0.9565
F-Beta 0.5 (precision):  0.9629


F-Beta 2.0 (recall):     0.9628
Training time:           5.0 sec
Prediction time:         0.2 sec

True Negative  (correctly 'dissatisfied'): 11491
False Positive (wrongly 'satisfied'):      249
False Negative (wrongly 'dissatisfied'):   520
True Positive  (correctly 'satisfied'):    8459


In [11]:
scaled_results.append(evaluate_model('K Neighbors (scaled)', KNeighborsClassifier(n_neighbors=5), X_train_scaled, X_test_scaled))

K Neighbors (scaled)
Accuracy:                0.9263
Precision (satisfied):   0.9482
Recall (satisfied):      0.8779
F1-Score:                0.9117
F-Beta 0.5 (precision):  0.9266


F-Beta 2.0 (recall):     0.9259
Training time:           0.1 sec
Prediction time:         0.6 sec

True Negative  (correctly 'dissatisfied'): 11309
False Positive (wrongly 'satisfied'):      431
False Negative (wrongly 'dissatisfied'):   1096
True Positive  (correctly 'satisfied'):    7883


In [12]:
scaled_results.append(evaluate_model('Logistic Regression (scaled)', LogisticRegression(max_iter=10000), X_train_scaled, X_test_scaled))

Logistic Regression (scaled)
Accuracy:                0.8720
Precision (satisfied):   0.8674
Recall (satisfied):      0.8319
F1-Score:                0.8493
F-Beta 0.5 (precision):  0.8718


F-Beta 2.0 (recall):     0.8718
Training time:           0.1 sec
Prediction time:         0.0 sec

True Negative  (correctly 'dissatisfied'): 10598
False Positive (wrongly 'satisfied'):      1142
False Negative (wrongly 'dissatisfied'):   1509
True Positive  (correctly 'satisfied'):    7470


## Comparison after scaling

In [13]:
pd.DataFrame(results + scaled_results)

,Model,Accuracy,Precision,Recall,F1,Train time,Predict time
0,Random Forest,0.962836,0.971186,0.942198,0.956473,5.083063,0.187304
1,K Neighbors,0.741445,0.719568,0.660987,0.689035,0.062959,0.604885
2,Logistic Regression,0.872002,0.867464,0.831718,0.849215,16.750859,0.001517
3,Random Forest (scaled),0.962884,0.971406,0.942087,0.956522,5.018924,0.180546
4,K Neighbors (scaled),0.926300,0.948160,0.877937,0.911698,0.056825,0.556222
5,Logistic Regression (scaled),0.872050,0.867394,0.831941,0.849298,0.111345,0.000387


## Conclusions

**Random Forest is the best model, 0.96.** It wins on all four metrics. Scaling does not change it at all, because trees split columns by thresholds and do not care about units.

**K Neighbors looked bad because of the data, not the algorithm.** Without scaling 0.74, with scaling 0.93. Nothing changed except the data preparation.

**Logistic Regression gives 0.87.** Scaling does not change the result, but training becomes much faster: 0.1 second instead of 16.6. It is a simple linear model, so it cannot catch complex patterns.

**All three models lose satisfied passengers more often than they add wrong ones.** Recall is lower than precision everywhere. Random Forest misses 519 and wrongly marks 251.

**Fast and accurate are not the same.** Random Forest trains slowly but predicts fast. K Neighbors trains instantly but predicts slowest, because it compares every new passenger with all 82 875 rows.

The prepared cells in `ML_general.ipynb` use `model.fit(X, y)`, which trains on the test data as well. Here all models use `model.fit(X_train, y_train)`.